# 02 — Extraction Run & Prompt Refinement

Interactively run extraction on the 20-passage held-out set, inspect outputs, iterate on the prompt, then run the full pipeline.

In [1]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
from IPython.display import display
from pathlib import Path

from src.extractor import get_held_out, run_extraction, run_extraction_batch, PASSAGES_CSV, EXTRACTIONS_CSV
from src.models import get_model
from src.config import CONFIG, reset_config

ROOT = Path('..')
OUTPUTS = ROOT / 'outputs'

In [ ]:
# If you edit config.yaml, run this to reload it without restarting the kernel
reset_config()
print("✓ Config reloaded! Current max_thinking_tokens:", CONFIG.get("max_thinking_tokens"))

## 1. Load passages and held-out set

In [2]:
passages = pd.read_csv(PASSAGES_CSV)
held_out = get_held_out(passages)
print(f'Passages total: {len(passages)} | Held-out: {len(held_out)}')
held_out[['passage_id', 'period', 'text']].head(5)

Passages total: 734 | Held-out: 10


,passage_id,period,text
0,p00559,post_covid,Participants agreed that near-term inflation p...
1,p00033,great_moderation,"Still, given the inherent uncertainty of budge..."
2,p00275,great_moderation,Although employment probably was not as weak a...
3,p00198,great_moderation,Business purchases of light vehicles picked up...
4,p00192,great_moderation,Core consumer price inflation eased somewhat b...


## 2. Run extraction on held-out set

Inspect the raw JSON outputs before running the full pipeline. Edit `src/prompts.py` and re-run this cell to iterate on the prompt.

In [3]:
# Change provider here if needed: 'gemini' | 'openai'
PROVIDER = 'gemini'  # None = use config.yaml default

held_out_extractions = await run_extraction_batch(held_out, provider=PROVIDER)

Extracting (concurrent): 100%|██████████| 10/10 [00:24<00:00,  2.45s/it]


In [ ]:
# Inspect results — scroll through and look for errors / bad extractions
display(held_out_extractions[[
    'passage_id', 'period', 'cause', 'connector', 'effect', 'hedge', 'direction'
]].to_string(index=False))

In [ ]:
held_out_extractions

In [ ]:
# Spot-check a single passage and its raw response
IDX = 0  # change this index to inspect different rows
row = held_out_extractions.iloc[IDX]
print('=== Passage ===')
print(passages[passages.passage_id == row.passage_id].iloc[0]['text'])
print('\n=== Raw LLM response ===')
try:
    print(json.dumps(json.loads(row.raw_response), indent=2))
except Exception:
    print(row.raw_response)

## 3. Run full extraction (after prompt is satisfactory)

Uncomment and run when you are happy with the prompt. Use `--resume` flag to continue a partial run.

In [ ]:
# full_extractions = run_extraction(passages, provider=PROVIDER, resume=True)
# print(f'Extracted {len(full_extractions)} triples from {len(passages)} passages')

## 4. Run LLM-as-a-judge (after extraction is complete)

In [ ]:
# from src.judge import run_judge
# extractions = pd.read_csv(EXTRACTIONS_CSV)
# judge_scores = run_judge(passages, extractions, provider=PROVIDER, resume=True)
# print(f'Judged {len(judge_scores)} triples')

## 5. Build annotation CSV

In [ ]:
# from src.build_annotation_csv import build_annotation_csv
# annotated = build_annotation_csv()
# print(f'Annotation CSV ready: {len(annotated)} rows')